## Step 5 — assign zone id to blocks dataset
**# of cells in notebook:** 1

**Purpose:** The blocks dataset does not nest perfectly within the zones dataset. A tighter fit may addressed in the future. This script matches block centroids to zone features from zones_1. The result is a blocks dataset where each block has a zone value. This allows us to visualize and analyze zones via the aggregation of blocks.   

**Input:**

- a geodatabase with: `zones_1`
- a geodatabase with: a blocks layer. The blocks layer form CIESIN has many columns. It's great, but also cumbsersome to work with. It would be wise to create a copy of that blocks layer but with the minimum columns necesary. The blocks layer needs to be projected to UTM (hence the block layer referenced in the script: `juba_blocks_20260415_small_utm36n` 

**Output:** `zones_1` with column `initial_zones_1_sj`

**Main logic:**

1. convert block to point, forcing centroid inside block polygon
2. spatial join points to zones
3. write `zone_1` ids back to blocks

In [ ]:
import arcpy
import os
import time
import traceback

# ============================================================
# USER INPUTS
# ============================================================

zones_1 = r"E:\World Bank deliverbale 1\_analysis\zones\zones.gdb\zones_1"

blocks = r"E:\World Bank deliverbale 1\_analysis\blocks\blocks.gdb\juba_blocks_20260415_small_utm36n"

# New field to create/update in blocks
out_field = "initial_zones_1_sj"


# ============================================================
# SETTINGS
# ============================================================

arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False

scratch_gdb = arcpy.env.scratchGDB

zones_tmp = os.path.join(scratch_gdb, "tmp_zones_1_with_oid")
block_centroids = os.path.join(scratch_gdb, "tmp_block_centroids_inside")
centroids_sj = os.path.join(scratch_gdb, "tmp_block_centroids_zones_sj")

zone_oid_tmp_field = "zone1_oid_tmp"


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def msg(text):
    print(text)
    arcpy.AddMessage(text)


def delete_if_exists(path):
    if arcpy.Exists(path):
        arcpy.management.Delete(path)


def check_exists(path, label):
    if not arcpy.Exists(path):
        raise FileNotFoundError(f"{label} does not exist:\n{path}")


def field_exists(table, field_name):
    return field_name.lower() in [f.name.lower() for f in arcpy.ListFields(table)]


def print_fields(table, label):
    msg(f"\nFields in {label}:")
    for f in arcpy.ListFields(table):
        msg(f"  {f.name} | {f.type}")


# ============================================================
# MAIN SCRIPT
# ============================================================

try:
    t0 = time.time()

    msg("Starting zone assignment to blocks using inside block centroids...")

    # --------------------------------------------------------
    # Check inputs
    # --------------------------------------------------------

    check_exists(zones_1, "zones_1")
    check_exists(blocks, "blocks")

    zones_desc = arcpy.Describe(zones_1)
    blocks_desc = arcpy.Describe(blocks)

    if zones_desc.shapeType != "Polygon":
        raise ValueError(f"zones_1 must be polygon, but it is: {zones_desc.shapeType}")

    if blocks_desc.shapeType != "Polygon":
        raise ValueError(f"blocks must be polygon, but it is: {blocks_desc.shapeType}")

    zones_oid_field = zones_desc.OIDFieldName
    blocks_oid_field = blocks_desc.OIDFieldName

    msg(f"\nzones_1: {zones_1}")
    msg(f"  Count: {arcpy.management.GetCount(zones_1)[0]}")
    msg(f"  OID field: {zones_oid_field}")
    msg(f"  Spatial reference: {zones_desc.spatialReference.name}")

    msg(f"\nblocks: {blocks}")
    msg(f"  Count: {arcpy.management.GetCount(blocks)[0]}")
    msg(f"  OID field: {blocks_oid_field}")
    msg(f"  Spatial reference: {blocks_desc.spatialReference.name}")

    if zones_desc.spatialReference.name != blocks_desc.spatialReference.name:
        msg("\nWARNING: zones_1 and blocks have different spatial references.")
        msg("For best results, project them to the same CRS before running this workflow.")
        msg(f"  zones_1 CRS: {zones_desc.spatialReference.name}")
        msg(f"  blocks CRS:  {blocks_desc.spatialReference.name}")

    # --------------------------------------------------------
    # Clean temporary outputs
    # --------------------------------------------------------

    msg("\nCleaning temporary outputs...")

    delete_if_exists(zones_tmp)
    delete_if_exists(block_centroids)
    delete_if_exists(centroids_sj)

    # --------------------------------------------------------
    # Copy zones_1 to scratch and preserve original OBJECTID
    # --------------------------------------------------------

    msg("\nCreating temporary copy of zones_1 with preserved OBJECTID...")

    arcpy.management.CopyFeatures(
        in_features=zones_1,
        out_feature_class=zones_tmp
    )

    if field_exists(zones_tmp, zone_oid_tmp_field):
        arcpy.management.DeleteField(zones_tmp, zone_oid_tmp_field)

    arcpy.management.AddField(
        in_table=zones_tmp,
        field_name=zone_oid_tmp_field,
        field_type="LONG"
    )

    arcpy.management.CalculateField(
        in_table=zones_tmp,
        field=zone_oid_tmp_field,
        expression=f"!{zones_oid_field}!",
        expression_type="PYTHON3"
    )

    msg(f"  Temporary zones created: {zones_tmp}")
    msg(f"  Preserved zones_1 OBJECTID in field: {zone_oid_tmp_field}")

    # --------------------------------------------------------
    # Create inside centroids from blocks
    # --------------------------------------------------------

    msg("\nCreating inside block centroid points...")

    arcpy.management.FeatureToPoint(
        in_features=blocks,
        out_feature_class=block_centroids,
        point_location="INSIDE"
    )

    centroid_count = int(arcpy.management.GetCount(block_centroids)[0])
    msg(f"  Created inside centroids: {block_centroids}")
    msg(f"  Centroid count: {centroid_count}")

    # FeatureToPoint creates ORIG_FID, which stores the source block OBJECTID.
    if not field_exists(block_centroids, "ORIG_FID"):
        print_fields(block_centroids, "block centroids")
        raise RuntimeError("Expected ORIG_FID field was not created on block centroids.")

    # --------------------------------------------------------
    # Spatial join centroids to temporary zones
    # --------------------------------------------------------

    msg("\nSpatial joining inside centroids to zones_1...")

    arcpy.analysis.SpatialJoin(
        target_features=block_centroids,
        join_features=zones_tmp,
        out_feature_class=centroids_sj,
        join_operation="JOIN_ONE_TO_ONE",
        join_type="KEEP_ALL",
        match_option="INTERSECT"
    )

    sj_count = int(arcpy.management.GetCount(centroids_sj)[0])
    msg(f"  Spatial join output: {centroids_sj}")
    msg(f"  Spatial join count: {sj_count}")

    if not field_exists(centroids_sj, "ORIG_FID"):
        print_fields(centroids_sj, "spatial join output")
        raise RuntimeError("Spatial join output is missing ORIG_FID.")

    if not field_exists(centroids_sj, zone_oid_tmp_field):
        print_fields(centroids_sj, "spatial join output")
        raise RuntimeError(f"Spatial join output is missing {zone_oid_tmp_field}.")

    # --------------------------------------------------------
    # Build dictionary: block OBJECTID -> zones_1 OBJECTID
    # --------------------------------------------------------

    msg("\nBuilding block-to-zone lookup...")

    block_to_zone = {}
    unmatched_centroids = 0

    with arcpy.da.SearchCursor(centroids_sj, ["ORIG_FID", zone_oid_tmp_field]) as cursor:
        for block_oid, zone_oid in cursor:
            if zone_oid is None:
                unmatched_centroids += 1
                block_to_zone[int(block_oid)] = None
            else:
                block_to_zone[int(block_oid)] = int(zone_oid)

    matched = sum(v is not None for v in block_to_zone.values())

    msg(f"  Block centroids with matched zone: {matched}")
    msg(f"  Block centroids without matched zone: {unmatched_centroids}")

    # --------------------------------------------------------
    # Add output field to blocks if needed
    # --------------------------------------------------------

    msg(f"\nPreparing output field in blocks: {out_field}")

    if not field_exists(blocks, out_field):
        arcpy.management.AddField(
            in_table=blocks,
            field_name=out_field,
            field_type="LONG"
        )
        msg(f"  Added field: {out_field}")
    else:
        msg(f"  Field already exists: {out_field}")
        msg("  Existing values will be overwritten.")

    # --------------------------------------------------------
    # Write zone OBJECTID values back to blocks
    # --------------------------------------------------------

    msg("\nWriting zones_1 OBJECTID values back to blocks...")

    updated = 0
    no_match = 0
    not_in_lookup = 0

    with arcpy.da.UpdateCursor(blocks, [blocks_oid_field, out_field]) as cursor:
        for block_oid, current_value in cursor:
            block_oid_int = int(block_oid)

            if block_oid_int not in block_to_zone:
                cursor.updateRow([block_oid, None])
                not_in_lookup += 1
                continue

            zone_oid = block_to_zone[block_oid_int]

            if zone_oid is None:
                cursor.updateRow([block_oid, None])
                no_match += 1
            else:
                cursor.updateRow([block_oid, zone_oid])
                updated += 1

    msg(f"  Updated blocks with zones_1 OBJECTID: {updated}")
    msg(f"  Blocks with no matched zone: {no_match}")
    msg(f"  Blocks missing from centroid lookup: {not_in_lookup}")

    # --------------------------------------------------------
    # Clean temporary outputs
    # --------------------------------------------------------

    msg("\nCleaning temporary outputs...")

    delete_if_exists(zones_tmp)
    delete_if_exists(block_centroids)
    delete_if_exists(centroids_sj)

    elapsed = round((time.time() - t0) / 60, 2)
    msg(f"\nDone. Elapsed time: {elapsed} minutes")

except Exception as e:
    msg("\nSCRIPT FAILED.")
    msg(str(e))
    msg(traceback.format_exc())
    raise